# 06 · Juntar todo: día y quebrada

Dos preguntas separadas que aquí se juntan: **¿qué días tienen condiciones para un huaico?** (modelo diario, cuaderno 05) y **¿qué quebradas participan cuando ocurre?** (historial por quebrada, cuaderno 03).

`P(quebrada i activa el día t) = P(día t con huaico) × π_i`, donde `π_i` es la fracción de los días con huaico ubicados en que aparece la quebrada.

> **No es un sistema de alerta ni una predicción para decidir evacuaciones.** Es un ejercicio de investigación para ver cuánto del catálogo explican el tiempo y el historial, con sus huecos a la vista. Ante una emergencia llama al 115 (INDECI).

Teorías que se prueban (escritas antes de mirar):
- **T9** el producto día × quebrada predice mejor *qué quebrada se activa qué día* que cada parte por separado;
- **T10** la participación `π_i` cambia en temporadas Niño (el catálogo de años Niño tiene más quebradas en cada día con huaico);
- **T11** el resultado no depende de los tres grandes racimos (1983, 1998, 2017).

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from illapa_research import filtro, integracion as ig, modelo_diario as md, nino, quebradas as q
from illapa_research.catalogo import CATALOGO, cargar_cuencas, cargar_todos

RAIZ = Path("..") if Path("../datos").exists() else Path(".")
D = md.cargar_datos(RAIZ)
print(D.inventario["puntos_era5"], D.inventario["cajas_sst"], D.inventario["ultimo_dia"])

## 1. Probabilidad diaria fuera de muestra

Se usa el modelo `lluvia+antecedentes` del cuaderno 05, dejando fuera una temporada cada vez. Cada día tiene una probabilidad que **no vio** su propia temporada.

In [ ]:
res = md.correr_teorias(D, "temporada", n_jobs=4, teorias=("lluvia+antecedentes",), con_oraculo=False)
X, y, ts, mes = md.matrices(D)
p_dia = pd.Series(res["lluvia+antecedentes"].p, index=y.index)
tab = md.tabla_metricas(res, y)
tab.round(3)

## 2. Participación de cada quebrada

`π_i` y su intervalo (Beta, 10-90 %), en todas las temporadas y separando Niño de las demás.

In [ ]:
cuencas = filtro.cuencas_a_tabla(cargar_cuencas())
refs = q.cargar_referencias(CATALOGO / "quebradas_referencias.csv")
eventos = cargar_todos(solo_dia_exacto=True, sin_baja_confianza=True)
hist = filtro.asignar_historial(eventos, refs, cuencas)
temporadas = D.temporadas
ranking = filtro.tabla(cuencas, hist, temporadas)
ubicadas = sorted(ranking.loc[ranking.evidencia == "fisica+historial", "slug"])
pi = ig.participacion(hist.activaciones, set(ubicadas), temporadas)
pi.pivot(index="slug", columns="temporadas", values="pi").round(2).sort_values("todas", ascending=False)

**T10.** ¿Hay más quebradas por día con huaico en los años Niño? (contando solo los días con al menos una quebrada ubicada)

In [ ]:
a = hist.activaciones[hist.activaciones.slug.isin(ubicadas)].drop_duplicates(["fecha", "slug"])
por_dia = a.groupby("fecha").agg(n=("slug", "nunique"), temporada=("temporada", "first"))
por_dia["nino"] = por_dia.temporada.isin(temporadas.index[temporadas.nino])
por_dia.groupby("nino").n.describe().round(2)

## 3. ¿Qué quebrada se activa qué día? Validación dejando una temporada fuera

Tres ponderaciones sobre todos los pares (quebrada, día de temporada de lluvias): solo el día, solo la quebrada, y el producto. `π_i` se aprende sin la temporada retenida.

In [ ]:
ts_dia = pd.Series(ts, index=y.index)
pares = ig.validar_pares(p_dia, hist.activaciones, ubicadas, ts_dia, temporadas)
pares.round(4)

**T11.** Sin los tres grandes racimos.

In [ ]:
mantener = ~np.isin(ts_dia, [1983, 1998, 2017])
pares_sin = ig.validar_pares(p_dia[mantener], hist.activaciones, ubicadas, ts_dia[mantener], temporadas)
pares_sin.round(4)

## Lectura

- El día del huaico lo explica el tiempo (lluvia y antecedentes); *qué quebrada* se activa lo explica sobre todo la presencia de gente que reporta (cuaderno 03). Si el producto no supera claramente a `solo_dia`, la parte «quebrada» aporta poco más que un reparto por exposición.
- Las cifras de este cuaderno cambian con los datos de ERA5 que existan en `datos/`; la versión que se corrió se anota en `docs/06-integracion.md`.